## 1. setup

In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Kaggle 上では /kaggle/input、ローカルではリポジトリ直下の data/ を使う
KAGGLE_INPUT = Path("/kaggle/input/competitions/spaceship-titanic")
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
    OUTPUT_DIR = Path("/kaggle/working")
else:
    PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    DATA_DIR = PROJECT_ROOT / "data"
    OUTPUT_DIR = PROJECT_ROOT / "submissions"
    OUTPUT_DIR.mkdir(exist_ok=True)

ID_COL = "PassengerId"
TARGET = "Transported"

SEED = 123
N_SPLITS = 5
THRESHOLD = 0.5
EARLY_STOPPING_ROUNDS = 100
LOG_PERIOD = 100

SPEND_SERVICES = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
LUXURY_SERVICES = ["RoomService", "Spa", "VRDeck"]
BASIC_SERVICES = ["FoodCourt", "ShoppingMall"]
CABINS = ["Deck", "Num", "Side"]
NAMES = ["FirstName", "FamilyName", "Name"]
CATEGORICAL_COLS = ["HomePlanet", "Destination", "VIP", "Deck", "Side", *NAMES]

# oof_acc 0.8159 -> 0.8148 に低下したため不採用
USE_LOGICAL_IMPUTATION = False

PARAMS = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.1,
    "num_leaves": 16,
    # 上限は大きく取り、実際の木の数は early stopping で決める
    "n_estimators": 100_000,
    "random_state": SEED,
    "importance_type": "gain",
}

## 2. load

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print(train.shape, test.shape)

In [ ]:
train.info()

In [ ]:
train.isnull().sum()

## 3. features

In [ ]:
def impute_by_logic(df):
    # コールドスリープ中の人は支出できない → 支出の欠損を0で埋める
    cryo = df["CryoSleep"] == 1
    df.loc[cryo, SPEND_SERVICES] = df.loc[cryo, SPEND_SERVICES].fillna(0)

    # 支出がある人はコールドスリープしていない → CryoSleep の欠損を0で埋める
    has_spend = df[SPEND_SERVICES].sum(axis=1) > 0
    df.loc[df["CryoSleep"].isna() & has_spend, "CryoSleep"] = 0.0
    return df


def split_name(df):
    return df["Name"].str.split(" ", expand=True).set_axis(["FirstName", "FamilyName"], axis=1)


def build_features(df, family_counts):
    df = df.copy()

    # NaN を残したいので astype(bool) ではなく map で数値化する
    df["CryoSleep"] = df["CryoSleep"].map({True: 1.0, False: 0.0})

    df[CABINS] = df["Cabin"].str.split("/", expand=True)
    df["Num"] = df["Num"].astype("float")

    if USE_LOGICAL_IMPUTATION:
        df = impute_by_logic(df)

    df["TotalSpend"] = df[SPEND_SERVICES].sum(axis=1)
    df["NoSpend"] = (df["TotalSpend"] == 0).astype(float)
    df["SpendCount"] = (df[SPEND_SERVICES] > 0).sum(axis=1)
    df["LuxurySpend"] = df[LUXURY_SERVICES].sum(axis=1)
    df["BasicSpend"] = df[BASIC_SERVICES].sum(axis=1)

    df[["FirstName", "FamilyName"]] = split_name(df)
    df["FamilySize"] = df["FamilyName"].map(family_counts).astype("float")

    # train と test で category の水準が揃わなくても、LightGBM は predict 時に学習時の水準へ読み替える
    df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].astype("category")
    return df


# 同じ家族が train と test に分かれて乗っているため、両方を合わせて数える
family_counts = split_name(pd.concat([train, test]))["FamilyName"].value_counts()

train = build_features(train, family_counts)
test = build_features(test, family_counts)

## 4. dataset

In [ ]:
# ["FamilySize"] を追加: va 0.82 -> 0.81 に低下したため不採用
# ["NoSpend", "SpendCount"] を追加: va 変化なし、LB 0.806 -> 0.805 に低下したため不採用
# "VIP" を除外: va 0.815+-0.004
FEATURES = [
    "Age",
    "TotalSpend",
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    *CABINS,
    *SPEND_SERVICES,
]  # va: 0.816+-0.005

x_train, y_train = train[FEATURES], train[TARGET]
x_test = test[FEATURES]
x_train.info()

## 5. model train

In [ ]:
def train_cv(x, y, params, folds):
    models = []
    oof = np.zeros(len(x))
    scores = []
    importances = []

    for nfold, (idx_tr, idx_va) in enumerate(folds):
        print("-" * 20, nfold, "-" * 20)
        x_tr, y_tr = x.iloc[idx_tr], y.iloc[idx_tr]
        x_va, y_va = x.iloc[idx_va], y.iloc[idx_va]
        print(f"y: {y.mean():.3f}, y_tr: {y_tr.mean():.3f}, y_va: {y_va.mean():.3f}")

        model = lgb.LGBMClassifier(**params)
        model.fit(
            x_tr,
            y_tr,
            eval_set=[(x_tr, y_tr), (x_va, y_va)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True),
                lgb.log_evaluation(LOG_PERIOD),
            ],
        )
        models.append(model)
        oof[idx_va] = model.predict_proba(x_va)[:, 1]

        acc_tr = accuracy_score(y_tr, model.predict(x_tr))
        acc_va = accuracy_score(y_va, model.predict(x_va))
        print(f"[accuracy] tr: {acc_tr:.2f}, va: {acc_va:.2f}")
        scores.append({"nfold": nfold, "tr": acc_tr, "va": acc_va})

        importances.append(pd.DataFrame({"col": x.columns, "imp": model.feature_importances_, "nfold": nfold}))

    scores = pd.DataFrame(scores)
    importance = (
        pd.concat(importances, ignore_index=True)
        .groupby("col")["imp"]
        .agg(imp="mean", imp_std="std")
        .sort_values("imp", ascending=False)
        .reset_index()
    )
    return models, oof, scores, importance

In [ ]:
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(x_train, y_train))
models, oof, scores, importance = train_cv(x_train, y_train, PARAMS, folds)

print("-" * 20, "result", "-" * 20)
display(scores)
print(
    f"[cv] tr {scores['tr'].mean():.3f}+-{scores['tr'].std(ddof=0):.3f}",
    f"[cv] va {scores['va'].mean():.3f}+-{scores['va'].std(ddof=0):.3f}",
)
oof_acc = accuracy_score(y_train, oof > THRESHOLD)
print(f"[oof] accuracy: {oof_acc:.4f}")

In [ ]:
importance

## 6. model inference

In [ ]:
# fold ごとの予測ラベルの多数決ではなく確率を平均する。多数決は5票のため境界付近の情報を捨てる
y_test_proba = np.mean([m.predict_proba(x_test)[:, 1] for m in models], axis=0)

df_submit = pd.DataFrame({ID_COL: test[ID_COL], TARGET: y_test_proba > THRESHOLD})
display(df_submit.head(5))
df_submit.to_csv(OUTPUT_DIR / "submission.csv", index=False)